# 12회차 실습 노트북 — 핀테크 데이터 시각화

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adminmrobo/fintech_data_analysis/blob/main/notebooks/12_visualization.ipynb)

해설 페이지: [https://adminmrobo.github.io/fintech_data_analysis/12_visualization.html](https://adminmrobo.github.io/fintech_data_analysis/12_visualization.html)

위에서부터 셀을 차례로 실행하세요(Shift+Enter). 데이터는 저장소 `data/` 폴더(fintech-data 복사본)에서 읽으며, 무작위 시드를 42로 고정해 해설 페이지의 대결 숫자와 같은 값이 나옵니다.

글 · 안상선 ((주)M-Robo 대표)

In [1]:
# 준비: 라이브러리와 데이터 불러오기 (Colab에서는 그대로 실행하면 됩니다)
import os, urllib.parse, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

BASE = 'https://raw.githubusercontent.com/adminmrobo/fintech_data_analysis/main/data/'
def load(path, **kw):
    """저장소의 data/ 폴더에서 CSV를 읽는다. 내 컴퓨터에 저장소가 있으면 그 파일을, 없으면 GitHub에서 읽는다."""
    for root in ('data', '../data'):
        f = os.path.join(root, path)
        if os.path.exists(f):
            return pd.read_csv(f, **kw)
    return pd.read_csv(BASE + urllib.parse.quote(path), **kw)

RS = 42   # 해설 페이지와 같은 숫자가 나오도록 무작위 시드를 고정

## 1. 꼬리가 긴 분포: 보험료

In [2]:
ins = load('Revolution/insurance.csv')
plt.hist(ins.charges, bins=40); plt.axvline(ins.charges.mean(), c='r', label='mean'); plt.axvline(ins.charges.median(), c='k', label='median')
plt.legend(); plt.xlabel('charges ($)'); plt.show()
print('평균', round(ins.charges.mean()), ' 중앙값', round(ins.charges.median()), ' 왜도', round(ins.charges.skew(), 2))

평균 13270  중앙값 9382  왜도 1.52


## 2. 대결 재현: 숫자 하나로 새 고객 보험료 맞히기 — 평균 vs 중앙값
잣대(MAE, RMSE)에 따라 승자가 바뀝니다.

In [3]:
r = np.random.default_rng(RS); i = r.permutation(len(ins))
a, c = ins.charges.values[i[:900]], ins.charges.values[i[900:]]
for name, v in [('평균', a.mean()), ('중앙값', np.median(a))]:
    print(f'{name:4s} {v:8.0f}   MAE {np.abs(c - v).mean():8.0f}   RMSE {np.sqrt(((c - v) ** 2).mean()):8.0f}')

평균      13165   MAE     9603   RMSE    12720
중앙값      9413   MAE     8859   RMSE    13353


## 3. 상관 행렬: 수익률로 볼 것

In [4]:
p = load('Revolution3/portfolio_data.csv')
ret = p[['AMZN', 'DPZ', 'BTC', 'NFLX']].pct_change().dropna()
print('가격 상관\n', p[['AMZN', 'DPZ', 'BTC', 'NFLX']].corr().round(2))
print('\n수익률 상관\n', ret.corr().round(2))

가격 상관
       AMZN   DPZ   BTC  NFLX
AMZN  1.00  0.97  0.76  0.98
DPZ   0.97  1.00  0.71  0.95
BTC   0.76  0.71  1.00  0.75
NFLX  0.98  0.95  0.75  1.00

수익률 상관
       AMZN   DPZ   BTC  NFLX
AMZN  1.00  0.31 -0.01  0.44
DPZ   0.31  1.00  0.02  0.25
BTC  -0.01  0.02  1.00  0.00
NFLX  0.44  0.25  0.00  1.00


**AI 판단 연습** — 가격 상관이 0.9를 넘는데 수익률 상관은 0.2 안팎이라면, "두 종목이 같이 움직인다"는 말은 어느 쪽 숫자에 근거해야 할까요?